# How large is the learned stride-4 correction

# Imports

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch import nn

from uavloc.data.annotated import IGNORE_LABEL
from uavloc.features.imagenet import normalise
from uavloc.features.segmentation import CLASSES, LRASPPLogits, load_segmenter
from uavloc.segmentation.eval import load_val_sets
from uavloc.track import REPO_ROOT

# Params

In [ ]:
CHECKPOINT = "pocs/lraspp-stride4-decoder/model-stride4-seed42.pt"
INIT_CHECKPOINT = "pocs/mobilenet-lraspp-speed/model-lraspp-bn-satellite.pt"
DECODER_WIDTH = 32
WINDOWS = [["bushcha", 0, 1664, 1472], ["birky-strip", 0, 64, 256], ["bushcha", 1, 1024, 1024]]
WINDOW_PX = 512

In [ ]:
BUILDING = CLASSES.index("building")
STRIDE4_LAYER, LOW_LAYER, HIGH_LAYER = "3", "4", "16"

# Model

In [ ]:
class Stride4LRASPP(nn.Module):
    """Same as decoder.ipynb: output = up_x8(stride-8 logits) + up_x4(stride-4 correction)."""

    def __init__(self, base: LRASPPLogits, width: int) -> None:
        super().__init__()
        self.base = base
        self.decoder = nn.Sequential(nn.Conv2d(24 + len(CLASSES), width, 3, padding=1, bias=False),
                                     nn.BatchNorm2d(width), nn.ReLU(inplace=True))
        self.correction = nn.Conv2d(width, len(CLASSES), 1)

    def logits_and_correction(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        feats, h = {}, x
        for name, module in self.base.net.backbone.items():
            h = module(h)
            if name in (STRIDE4_LAYER, LOW_LAYER, HIGH_LAYER):
                feats[name] = h
        logits8 = self.base.net.classifier({"low": feats[LOW_LAYER], "high": feats[HIGH_LAYER]})
        up = F.interpolate(logits8, size=feats[STRIDE4_LAYER].shape[-2:], mode="bilinear", align_corners=False)
        return logits8, self.correction(self.decoder(torch.cat([feats[STRIDE4_LAYER], up], 1)))

In [ ]:
model = Stride4LRASPP(load_segmenter(REPO_ROOT / INIT_CHECKPOINT, "cpu", architecture="lraspp"), DECODER_WIDTH)
model.load_state_dict(torch.load(REPO_ROOT / CHECKPOINT, map_location="cpu"))
model = model.eval()

# Data

In [ ]:
val_sets = load_val_sets()

# Run

In [ ]:
pd.DataFrame({name: {"norm": p.norm().item(), "abs_max": p.abs().max().item()}
              for name, p in model.named_parameters() if not name.startswith("base.")}).T.round(4)

In [ ]:
@torch.no_grad()
def window_outputs(name: str, index: int, top: int, left: int) -> dict:
    """Full-res base logits, full-res correction and the label for one `WINDOW_PX` window."""
    image = val_sets[name][index]
    win = np.s_[top:top + WINDOW_PX, left:left + WINDOW_PX]
    x = torch.from_numpy(normalise(image.rgb[win]).transpose(2, 0, 1).copy())[None]
    logits8, correction4 = model.logits_and_correction(x)
    up = lambda t: F.interpolate(t, size=(WINDOW_PX, WINDOW_PX), mode="bilinear", align_corners=False)[0]
    return {"base": up(logits8), "correction": up(correction4), "label": image.label[win], "rgb": image.rgb[win]}

In [ ]:
def correction_stats(out: dict) -> dict:
    """Size of the class-relative correction against the base top-1 minus top-2 margin, and pixels it flips."""
    base, corr = out["base"], out["correction"] - out["correction"].mean(0, keepdim=True)
    top2 = base.topk(2, 0).values
    flipped = (base + out["correction"]).argmax(0) != base.argmax(0)
    return {"corr_abs_median": corr.abs().amax(0).median().item(), "corr_abs_p99": corr.abs().amax(0).quantile(.99).item(),
            "margin_median": (top2[0] - top2[1]).median().item(), "flipped_px": int(flipped.sum()),
            "flipped_frac": flipped.float().mean().item()}

In [ ]:
outputs = [window_outputs(*w) for w in WINDOWS]
pd.DataFrame([correction_stats(o) for o in outputs], index=[f"{w[0]}[{w[1]}]" for w in WINDOWS]).round(4)

# Plots

In [ ]:
fig, axes = plt.subplots(len(WINDOWS), 3, figsize=(12, 4 * len(WINDOWS)))
for row, out in zip(axes, outputs):
    corr = out["correction"] - out["correction"].mean(0, keepdim=True)
    row[0].imshow(out["rgb"]); row[0].set_title("RGB")
    im = row[1].imshow(corr.abs().amax(0), cmap="magma"); row[1].set_title("max_c |correction| (class-relative)")
    plt.colorbar(im, ax=row[1], fraction=.046)
    im = row[2].imshow(corr[BUILDING], cmap="coolwarm", vmin=-corr[BUILDING].abs().max(), vmax=corr[BUILDING].abs().max())
    row[2].set_title("building correction"); plt.colorbar(im, ax=row[2], fraction=.046)
    for ax in row:
        ax.axis("off")
fig.tight_layout()